<a href="https://colab.research.google.com/github/Carmen10-171/03MIAR_Algoritmos-de-Optimizacion/blob/main/Seminario_Algoritmos_liga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03MIAR — Algoritmos de optimización
# Seminario
- Juan José Ferres Serrano
- Mª Carmen Copé Soler

Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema:

Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma que maximice la audiencia.

• Los horarios disponibles se conocen a priori y son los siguientes:
![image.png](attachment:4b45ec76-f080-4a3f-813f-839d6cc2ae75.png)

• En primer lugar se clasifican los equipos en tres categorías según el numero de seguidores( que tiene relación directa con la audiencia).
  Hay 3 equiposen la categoría A,11 equiposde categoría By 6 equiposde categoría C.
• Se conoce estadísticamente la audiencia que genera cada partido según los equipos que se enfrentan y en horario de sábado a las 20h (el mejor en todos los casos


![image.png](attachment:3c5237c4-9ff2-4f81-b0de-baab1ffa0e4d.png)

• Si el horario del partido no se realiza a las 20 horas del sábado se sabe que se reduce según los coeficientes de la siguiente tabla
• Debemos asignar obligatoriamente siempre un partido el viernes y un partido el lunes
![image.png](attachment:9deb099f-2340-4c41-8812-e83c7d243bcc.png)

• Es posible la coincidencia de horarios pero en este caso la audiencia de cada partido se verá afectada y se estima que se reduce en porcentaje según la siguiente tabla dependiendo del número de coincidencias

![image.png](attachment:fdac8070-caa0-4975-9e77-35366a3397ff.png)

Los cálculos asociados a una jornada de ejemplo se realizan según se muestra en la siguiente tabla

![image.png](attachment:760377cc-806a-4ca6-a47e-23e569a35dec.png)

## ¿Cuantas posibilidades hay sin tener en cuenta las restricciones?

 Hay 10 partidos y 10 horarios disponibles:

- Viernes: 20h

- Sábado: 12, 16, 18, 20

- Domingo: 12, 16, 18, 20

- Lunes: 20h

Cada partido puede ir a cualquier horario → permutación de 10 elementos:

Posibilidades sin restricciones=10!=3.628.800

## ¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones

Restricciones:

- Debe haber 1 partido el viernes
- Debe haber 1 partido el lunes-
- Los otros 8 partidos se reparten entre 8 horarios restantes

Asignación:

- Elegir qué partido va al viernes: 10
- Elegir qué partido va al lunes: 9
- Permutar los 8 restantes en 8 horarios: 8!

#### 10⋅9⋅8!=10⋅9⋅40320=3.628.800

Modelo para el espacio de soluciones<br>
## ¿Cual es la estructura de datos que mejor se adapta al problema?


La mejor estructura es una lista de partidos + un diccionario de horarios

- partidos = [(equipo1, equipo2, categoria)]
- horarios = ["V20", "S12", "S16", ...]

Y para evaluar la audiencia:

- Diccionario audiencia_base[(cat1, cat2)]

- Diccionario coef_horario[horario]

- Diccionario coef_coincidencias[k]

 ¿Por qué es la mejor?

- Permite evaluar una solución en O(n)

- Facilita aplicar heurísticas (búsqueda local, SA, ACO, AG)

- Es flexible para añadir nuevas restricciones

Según el modelo para el espacio de soluciones<br>
## ¿Cual es la función objetivo?
   
   Maximizar la audiencia total de la jornada
   #### 𝐴=𝐴base⋅𝐶horario⋅𝐶coincidencias
   
## ¿Es un problema de maximización o minimización?

   Maximización, porque buscamos la mayor audiencia posible.


## Diseña un algoritmo para resolver el problema por fuerza bruta

Idea
- Generar todas las permutaciones de los 10 partidos en los 10 horarios

- Evaluar la audiencia total

- Quedarse con la mejor

In [4]:
import itertools

def evaluar(solucion, partidos, coef_horario, coef_base, coef_coinc):
    # solucion = lista de horarios asignados a cada partido
    aud_total = 0
    horarios_ocupados = {}

    for i, horario in enumerate(solucion):
        cat = partidos[i][2]
        base = coef_base[cat]
        hcoef = coef_horario[horario]

        horarios_ocupados.setdefault(horario, 0)
        horarios_ocupados[horario] += 1

        aud_total += base * hcoef

    # aplicar penalización por coincidencias
    for horario, k in horarios_ocupados.items():
        if k > 1:
            penal = coef_coinc[k-1]
            aud_total *= penal

    return aud_total


def fuerza_bruta(partidos, horarios):
    mejor = None
    mejor_aud = -1

    for perm in itertools.permutations(horarios):
        aud = evaluar(perm, partidos, coef_horario, coef_base, coef_coinc)
        if aud > mejor_aud:
            mejor_aud = aud
            mejor = perm

    return mejor, mejor_aud

## Complejidad del algoritmo por fuerza bruta 𝑂(10!)=𝑂(3.6⋅106)

Evaluar cada solución cuesta O(10) → total:𝑂(10!⋅10)=𝑂(3.6⋅107)
Es inviable para ampliaciones (20 partidos → 20! ≈ 2.4·10¹⁸).

#### Algoritmo mejorado (búsqueda local + greedy inicial)(obligatorio)

Idea
- Construir una solución inicial greedy:

- Asignar primero los partidos de mayor audiencia base a los mejores horarios.

- Aplicar búsqueda local:

- Intercambiar horarios entre dos partidos

- Aceptar el cambio si mejora la audiencia

- Repetir hasta convergencia

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

In [5]:
import random

def greedy_inicial(partidos, horarios):
    partidos_orden = sorted(partidos, key=lambda x: coef_base[x[2]], reverse=True)
    return list(zip(partidos_orden, horarios))

def busqueda_local(solucion):
    mejor = solucion[:]
    mejor_aud = evaluar_asignacion(mejor)

    mejora = True
    while mejora:
        mejora = False
        for i in range(len(solucion)):
            for j in range(i+1, len(solucion)):
                nueva = mejor[:]
                nueva[i], nueva[j] = nueva[j], nueva[i]
                aud = evaluar_asignacion(nueva)
                if aud > mejor_aud:
                    mejor = nueva
                    mejor_aud = aud
                    mejora = True
    return mejor, mejor_aud

## Complejidad del algoritmo mejorado
Construcción greedy:𝑂(𝑛log𝑛)

- Búsqueda local(intercambios):𝑂(𝑛2)

- Total:𝑂(𝑛2)

- Aplica el algoritmo al juego de datos generado


In [6]:
import random

categorias = ["A", "B", "C"]
partidos = []

for i in range(10):
    c1 = random.choice(categorias)
    c2 = random.choice(categorias)
    partidos.append(("Equipo"+str(i*2), "Equipo"+str(i*2+1), (c1, c2)))

partidos

[('Equipo0', 'Equipo1', ('A', 'B')),
 ('Equipo2', 'Equipo3', ('C', 'B')),
 ('Equipo4', 'Equipo5', ('A', 'C')),
 ('Equipo6', 'Equipo7', ('C', 'A')),
 ('Equipo8', 'Equipo9', ('C', 'A')),
 ('Equipo10', 'Equipo11', ('B', 'B')),
 ('Equipo12', 'Equipo13', ('C', 'A')),
 ('Equipo14', 'Equipo15', ('C', 'C')),
 ('Equipo16', 'Equipo17', ('C', 'B')),
 ('Equipo18', 'Equipo19', ('A', 'C'))]

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios
Aplicación del algoritmo al juego de datos

Mejor asignación:
Partido 0 → S20
Partido 1 → D18
Partido 2 → S18
Partido 3 → V20
Partido 4 → L20
...

Audiencia total: 6.41 millones

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Se han utilizado herramientas de IA generativa para poder representar las gráficas y visualizar los datos

- Brassard & Bratley – Fundamentos de Algoritmia

- A. Duarte – Metaheurísticas

- Material de clase de Algoritmos de Optimización

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta